# Chapter 8 — Anisotropic and Asymmetric Extensions

Reproduces:
- Figure 8.1: Bilinear Q/K asymmetric Gram matrix vs its transpose.
- Figure 8.2: KL-divergence kernel — asymmetric.
- Figure 8.3: Directed graph Laplacian eigenvectors.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.asymmetric import (
    BilinearQKKernel, BregmanKernel, kl_kernel, squared_euclidean_kernel,
    DirectedLaplacianKernel,
)

torch.manual_seed(42); np.random.seed(42)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Figure 8.1: Bilinear Q/K asymmetry

In [ ]:
X = torch.randn(15, 4)
k = BilinearQKKernel(d_in=4, d_h=4, mode='inner')
with torch.no_grad():
    K = k(X, X)
    K_S = (K + K.T) / 2
    K_A = (K - K.T) / 2
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
vmax = K.abs().max().item()
for ax, M, title in zip(axes, [K, K_S, K_A], ['$W$', '$W_S = (W + W^T)/2$', '$W_A = (W - W^T)/2$']):
    im = ax.imshow(M.numpy(), cmap='RdBu', vmin=-vmax, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax)
fig.suptitle('Figure 8.1: Bilinear Q/K kernel decomposition')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_08_01_bilinear_qk.pdf', bbox_inches='tight')
plt.show()

## Figure 8.2: KL divergence (asymmetric Bregman)

In [ ]:
X = torch.softmax(torch.randn(10, 4), dim=-1)
k_kl = kl_kernel()
k_eu = squared_euclidean_kernel()
D_kl = k_kl.divergence(X, X)
D_eu = k_eu.divergence(X, X)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, D, name in zip(axes, [D_eu, D_kl], ['Sq Euclidean (symmetric)', 'KL divergence (asymmetric)']):
    im = ax.imshow(D.numpy(), cmap='viridis')
    sym_diff = (D - D.T).abs().max().item()
    ax.set_title(f'{name}\nmax|D - D^T| = {sym_diff:.3f}')
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax)
fig.suptitle('Figure 8.2: Bregman divergences — squared Euclidean vs KL')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_08_02_bregman.pdf', bbox_inches='tight')
plt.show()

## Figure 8.3: Directed graph Laplacian

In [ ]:
torch.manual_seed(0)
W_asym = torch.randn(15, 15).abs() + 0.1
L_dir = DirectedLaplacianKernel()(W_asym)
eigvals = torch.linalg.eigvalsh(L_dir)
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.plot(eigvals.numpy(), 'o-')
ax.set_xlabel('eigenvalue index'); ax.set_ylabel('lambda')
ax.set_title('Figure 8.3: Eigenvalues of directed Laplacian on asymmetric W')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_08_03_directed_laplacian.pdf', bbox_inches='tight')
plt.show()
print(f'L_dir is symmetric: {(L_dir - L_dir.T).abs().max().item() < 1e-5}')
print(f'Smallest eigenvalue: {eigvals.min().item():.4f} (PSD: {eigvals.min().item() > -1e-4})')